# ListT5 Combined Grouping and BM25 Top-K Experiment

This notebook combines the two lightweight experiment directions:

1. **Grouping comparison** at paper-style BM25 `topk=100`: original sequential grouping vs score-balanced grouping.
2. **Candidate-budget comparison** with sequential grouping only: BM25 `topk=40`, `60`, and `80`.

The defaults keep the paper's inference settings (`listwise_k=5`, `out_k=2`, `rerank_topk=10`) and use `MAX_QUERIES=50` so the run is still practical on Kaggle. Because this is a 50-query subset, the Table 2 comparison is a sanity reference, not an exact paper reproduction.

## 1. Setup

Run this on Kaggle with GPU enabled and Internet enabled. This intentionally uses normal `pip install` dependencies without pinning old `transformers`, because old `tokenizers` wheels can fail on current Kaggle Python.

In [ ]:
!pip install -q jsonlines sentencepiece huggingface_hub beir

## 2. Clone and Import ListT5

This cell clones the official codebase if it is not already present, then applies small compatibility patches needed by newer Kaggle `torch` / `transformers` versions.

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import subprocess
import sys
import time
from types import SimpleNamespace

import pandas as pd

PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
LISTT5_ROOT = PROJECT_ROOT / 'ListT5'

if not (LISTT5_ROOT / 'run_listt5.py').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/soyoung97/ListT5.git', str(LISTT5_ROOT)],
        check=True,
    )

assert (LISTT5_ROOT / 'run_listt5.py').exists(), f'Could not find run_listt5.py at {LISTT5_ROOT}'
sys.path.insert(0, str(LISTT5_ROOT))

import torch
import FiDT5 as fid_module
from beir_eval import run_rerank_eval
from beir_length_mapping import BEIR_LENGTH_MAPPING
from run_listt5 import ListT5Evaluator, read_jsonl

# Compatibility patch 1: newer transformers expects encoder.embed_tokens.
if not hasattr(fid_module.EncoderWrapper, 'embed_tokens'):
    fid_module.EncoderWrapper.embed_tokens = property(lambda self: self.encoder.embed_tokens)

# Compatibility patch 2: newer T5 encoder calls pass more positional args than
# the original CheckpointWrapper.forward accepted.
def _checkpoint_wrapper_forward_compat(self, *args, **kwargs):
    if self.use_checkpoint and self.training:
        return torch.utils.checkpoint.checkpoint(
            lambda *inner_args: self.module(*inner_args, **kwargs),
            *args,
        )
    return self.module(*args, **kwargs)

fid_module.CheckpointWrapper.forward = _checkpoint_wrapper_forward_compat

DATA_DIR = PROJECT_ROOT / 'data' / 'beir-eval-bm25-top100'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'combined_grouping_topk'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LIVE_RESULTS_CSV = OUTPUT_DIR / 'results_live.csv'
LIVE_RESULTS_TXT = OUTPUT_DIR / 'results_live.txt'
SUMMARY_CSV = OUTPUT_DIR / 'combined_summary.csv'
SUMMARY_TXT = OUTPUT_DIR / 'combined_summary.txt'

print('Project root:', PROJECT_ROOT, flush=True)
print('ListT5 root:', LISTT5_ROOT, flush=True)
print('Data dir:', DATA_DIR, flush=True)
print('Output dir:', OUTPUT_DIR, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0), flush=True)

## 3. Experiment Configuration

Edit this cell to choose datasets or expand the grid. Current defaults:

- grouping comparison: sequential `topk=100` and score-balanced `topk=100`
- top-k comparison: sequential only for `topk=40`, `60`, `80`
- `MAX_QUERIES=50`

In [ ]:
MODEL_PATH = 'Soyoung97/ListT5-base'
HF_DATASET_REPO = 'Soyoung97/beir-eval-bm25-top100'

# Edit this list to choose the datasets for the run.
DEFAULT_DATASETS = [
    'nfcorpus',
    'scifact',
    'arguana',
    'scidocs',
    'fiqa',
]

# Use the first 50 queries for a practical early experiment. Set to None for full evaluation.
MAX_QUERIES = 50

# Paper-style ListT5-base settings.
LISTWISE_K = 5
OUT_K = 2
RERANK_TOPK = 10
# This is only the inference mini-batch size, not a paper algorithm setting.
# Keep 20 for the usual 512-token BEIR datasets, matching the earlier notebook.
BATCH_SIZE = 20

# Controlled low-resource setting: cap every dataset at 512 tokens.
# This keeps all five datasets comparable and avoids ArguAna's default 1024-token cost.
MAX_INPUT_LENGTH = 512
SEED = 0

# Plain print progress is useful in Kaggle "Save & Run All" logs.
PRINT_EVERY_FORWARDS = 20

# Experiment 1: grouping method at BM25 top-100.
GROUPING_EXPERIMENTS = [
    {'strategy': 'sequential', 'topk': 100},
    {'strategy': 'score_balanced', 'topk': 100},
]

# Experiment 2: candidate-budget / top-k method. Keep sequential only for now.
# To later test score-balanced at smaller top-k values, add entries such as:
# {'strategy': 'score_balanced', 'topk': 60}
TOPK_EXPERIMENTS = [
    {'strategy': 'sequential', 'topk': 40},
    {'strategy': 'sequential', 'topk': 60},
    {'strategy': 'sequential', 'topk': 80},
]

# Paper Table 2, BM25 top-100, ListT5-base (r=2), NDCG@10.
TABLE2_LISTT5_BASE_TOP100 = {
    'trec-covid': 0.783,
    'nfcorpus': 0.356,
    'bioasq': 0.564,
    'nq': 0.531,
    'hotpotqa': 0.726,
    'fiqa': 0.396,
    'signal': 0.335,
    'news': 0.485,
    'robust04': 0.521,
    'arguana': 0.489,
    'touche': 0.334,
    'cqadupstack': 0.388,
    'quora': 0.864,
    'dbpedia-entity': 0.437,
    'scidocs': 0.176,
    'fever': 0.798,
    'climate-fever': 0.240,
    'scifact': 0.741,
}

random.seed(SEED)

config_preview = pd.DataFrame(
    [{'experiment_kind': 'grouping', **row} for row in GROUPING_EXPERIMENTS]
    + [{'experiment_kind': 'topk', **row} for row in TOPK_EXPERIMENTS]
)
print('Datasets:', DEFAULT_DATASETS, flush=True)
print('Max queries:', MAX_QUERIES, flush=True)
config_preview

## 4. Dataset Helper

The input files are BM25 top-100 JSONL files from the same Hugging Face dataset used by the ListT5 codebase. For `MAX_QUERIES=50`, the notebook writes a stable subset file so every approach uses the exact same queries.

In [ ]:
def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')


def dataset_path(dataset_name, max_queries=MAX_QUERIES):
    full_path = DATA_DIR / f'{dataset_name}.jsonl'
    bundled_path = LISTT5_ROOT / f'{dataset_name}.jsonl'

    if not full_path.exists():
        if bundled_path.exists():
            full_path = bundled_path
        else:
            from huggingface_hub import hf_hub_download
            downloaded = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                filename=f'{dataset_name}.jsonl',
                repo_type='dataset',
                local_dir=str(DATA_DIR),
            )
            full_path = Path(downloaded)

    if max_queries is None:
        return full_path

    subset_tag = f'maxq{max_queries}'
    subset_path = DATA_DIR / subset_tag / f'{dataset_name}.jsonl'
    if not subset_path.exists():
        rows = read_jsonl(str(full_path))[:max_queries]
        write_jsonl(subset_path, rows)
        print(f'[data] wrote {len(rows)} rows -> {subset_path}', flush=True)
    else:
        print(f'[data] reuse subset -> {subset_path}', flush=True)
    return subset_path


for dataset_name in DEFAULT_DATASETS:
    path = dataset_path(dataset_name)
    print(dataset_name, 'queries=', len(read_jsonl(str(path))), 'path=', path, flush=True)

## 5. Grouping Policies

`sequential` matches the official ListT5 grouping behavior. `score_balanced` spreads BM25 ranks across groups so every first-round group gets a mix of strong and weaker BM25 candidates.

In [ ]:
def sequential_groups(items, group_size, seed=0):
    items = list(items)
    return [items[i:i + group_size] for i in range(0, len(items), group_size)]


def score_balanced_groups(items, group_size, seed=0):
    items = sorted(list(items))
    if len(items) <= group_size:
        return [items]

    n_groups = math.ceil(len(items) / group_size)
    groups = [[] for _ in range(n_groups)]
    for i, item in enumerate(items):
        groups[i % n_groups].append(item)
    return [group for group in groups if group]


GROUPING_POLICIES = {
    'sequential': sequential_groups,
    'score_balanced': score_balanced_groups,
}

demo = list(range(20))
pd.concat(
    [
        pd.DataFrame({'strategy': name, 'group': i + 1, 'indices': [group]})
        for name, fn in GROUPING_POLICIES.items()
        for i, group in enumerate(fn(demo, LISTWISE_K, seed=SEED))
    ],
    ignore_index=True,
)

## 6. Official Evaluator Subclass

This keeps the official ListT5 model loading, tournament sort, caching, and BEIR evaluation path. The only experiment hook is `group2chunks`, plus `args.topk` for the BM25 candidate budget.

In [ ]:
class CombinedListT5Evaluator(ListT5Evaluator):
    def load_model(self):
        start = time.time()
        print('Loading model..', flush=True)
        print(f'Loading fid model from {self.args.model_path}', flush=True)
        model = fid_module.FiDT5.from_pretrained(
            self.args.model_path,
            use_safetensors=False,
        ).to('cuda')
        model.eval()
        print(f'Done! took {time.time() - start:.2f} seconds', flush=True)
        return model

    def group2chunks(self, l, n=5):
        strategy = getattr(self.args, 'grouping_strategy', 'sequential')
        seed = getattr(self.args, 'seed', 0)
        if strategy not in GROUPING_POLICIES:
            raise ValueError(f'Unknown grouping strategy: {strategy}')
        yield from GROUPING_POLICIES[strategy](l, n, seed=seed)

    def run_inference(self, input_tensors):
        with torch.inference_mode():
            output = self.model.generate(
                **input_tensors,
                max_length=self.args.max_gen_length,
                return_dict_in_generate=True,
                output_scores=True,
            )
        self.num_forward += 1
        print_every = getattr(self.args, 'print_every_forwards', PRINT_EVERY_FORWARDS)
        if print_every and self.num_forward % print_every == 0:
            print(f'[progress] forward_calls={self.num_forward}', flush=True)
        return output


def make_args(dataset_name, strategy, topk, experiment_kind):
    input_path = dataset_path(dataset_name)
    subset_tag = 'allq' if MAX_QUERIES is None else f'maxq{MAX_QUERIES}'
    output_path = (
        OUTPUT_DIR
        / experiment_kind
        / strategy
        / f'topk{topk}'
        / subset_tag
        / f'{dataset_name}_output.jsonl'
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if dataset_name not in BEIR_LENGTH_MAPPING:
        raise ValueError(f'No BEIR_LENGTH_MAPPING entry for {dataset_name}')

    return SimpleNamespace(
        firststage_result_key='bm25_results',
        docid_key='docid',
        pid_key='pid',
        qrels_key='qrels',
        score_key='bm25_score',
        question_text_key='q_text',
        text_key='text',
        title_key='title',
        model_path=MODEL_PATH,
        topk=topk,
        max_input_length=MAX_INPUT_LENGTH,
        padding='max_length',
        listwise_k=LISTWISE_K,
        rerank_topk=RERANK_TOPK,
        out_k=OUT_K,
        dummy_number=21,
        verbose=False,
        seed=SEED,
        bsize=BATCH_SIZE,
        input_path=str(input_path),
        output_path=str(output_path),
        measure_flops=False,
        skip_no_candidate=False,
        skip_issubset=False,
        max_gen_length=LISTWISE_K + 2,
        grouping_strategy=strategy,
        experiment_kind=experiment_kind,
        print_every_forwards=PRINT_EVERY_FORWARDS,
    )

## 7. Run One Approach

Every completed approach saves into `results_live.csv` and `results_live.txt`. If a complete output JSONL already exists, the notebook reuses it and recomputes metrics only.

In [ ]:
def output_is_complete(output_path, input_path):
    output_path = Path(output_path)
    if not output_path.exists():
        return False
    try:
        return len(read_jsonl(str(output_path))) == len(read_jsonl(str(input_path)))
    except Exception:
        return False


def save_results_snapshot(rows):
    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    key_cols = ['experiment_kind', 'dataset', 'strategy', 'topk', 'max_queries']
    df = df.drop_duplicates(subset=key_cols, keep='last')
    df = df.sort_values(key_cols).reset_index(drop=True)
    LIVE_RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(LIVE_RESULTS_CSV, index=False)
    LIVE_RESULTS_TXT.write_text(df.to_string(index=False) + '\n', encoding='utf-8')
    print(f'[saved] live csv -> {LIVE_RESULTS_CSV}', flush=True)
    print(f'[saved] live txt -> {LIVE_RESULTS_TXT}', flush=True)
    return df


def clear_cuda_memory(label=''):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        free = free_bytes / (1024 ** 3)
        total = total_bytes / (1024 ** 3)
        prefix = f'[cuda cleanup {label}]' if label else '[cuda cleanup]'
        print(f'{prefix} allocated={allocated:.2f}GB reserved={reserved:.2f}GB free={free:.2f}/{total:.2f}GB', flush=True)


def run_single(dataset_name, strategy, topk, experiment_kind, reuse_existing=True):
    args = make_args(dataset_name, strategy=strategy, topk=topk, experiment_kind=experiment_kind)
    started = time.time()
    mode = 'new_inference'
    num_forward = None

    print('', flush=True)
    print(
        f'=== {experiment_kind} | {dataset_name} | {strategy} | topk={topk} | max_queries={MAX_QUERIES} ===',
        flush=True,
    )
    print(f'Model path: {args.model_path}', flush=True)
    print(f'Input: {args.input_path}', flush=True)
    print(f'Output: {args.output_path}', flush=True)
    print(f'Batch size: {args.bsize}', flush=True)
    print(f'Max input length: {args.max_input_length}', flush=True)

    clear_cuda_memory('before job')
    evaluator = None
    try:
        if reuse_existing and output_is_complete(args.output_path, args.input_path):
            print('[reuse] Complete output found. Recomputing metrics only.', flush=True)
            ndcg10, metric_text = run_rerank_eval(args.output_path)
            mode = 'reused_output'
        else:
            evaluator = CombinedListT5Evaluator(args)
            ndcg10, metric_text = evaluator.run_tournament_sort()
            num_forward = evaluator.num_forward
    except torch.cuda.OutOfMemoryError:
        print('[oom] CUDA out of memory. Freeing model/cache before raising. Try BATCH_SIZE=5 if this repeats.', flush=True)
        raise
    finally:
        if evaluator is not None:
            try:
                evaluator.model.to('cpu')
            except Exception:
                pass
            del evaluator
        clear_cuda_memory('after job')

    seconds = time.time() - started
    num_queries = len(read_jsonl(args.input_path))
    table2 = TABLE2_LISTT5_BASE_TOP100.get(dataset_name)
    row = {
        'experiment_kind': experiment_kind,
        'dataset': dataset_name,
        'strategy': strategy,
        'topk': topk,
        'max_queries': MAX_QUERIES,
        'ndcg@10': float(ndcg10),
        'table2_listt5_base_top100': table2,
        'delta_vs_table2_top100': None if table2 is None else float(ndcg10) - table2,
        'seconds': seconds,
        'seconds_per_query': seconds / num_queries if num_queries else None,
        'num_queries': num_queries,
        'num_forward': num_forward,
        'mode': mode,
        'output_path': str(args.output_path),
    }
    print('[done]', row, flush=True)
    return row

## 8. Run the Combined Grid

This runs all configured datasets through both experiment types. With the default 2 datasets, this is 10 jobs total:

- 2 grouping jobs per dataset: sequential top-100 and score-balanced top-100
- 3 top-k jobs per dataset: sequential top-40, top-60, top-80

In [ ]:
def build_jobs(datasets=DEFAULT_DATASETS):
    jobs = []
    for dataset_name in datasets:
        for spec in GROUPING_EXPERIMENTS:
            jobs.append({
                'experiment_kind': 'grouping',
                'dataset': dataset_name,
                'strategy': spec['strategy'],
                'topk': spec['topk'],
            })
        for spec in TOPK_EXPERIMENTS:
            jobs.append({
                'experiment_kind': 'topk',
                'dataset': dataset_name,
                'strategy': spec['strategy'],
                'topk': spec['topk'],
            })
    return jobs


def run_grid(datasets=DEFAULT_DATASETS):
    rows = []
    jobs = build_jobs(datasets)
    print(f'[grid] total_jobs={len(jobs)} datasets={datasets}', flush=True)

    for job_idx, job in enumerate(jobs, start=1):
        print('', flush=True)
        print(f'[grid] job {job_idx}/{len(jobs)} -> {job}', flush=True)
        row = run_single(
            dataset_name=job['dataset'],
            strategy=job['strategy'],
            topk=job['topk'],
            experiment_kind=job['experiment_kind'],
        )
        rows.append(row)
        live_df = save_results_snapshot(rows)
        display(live_df.tail(1))

    return save_results_snapshot(rows)


results = run_grid()
results

## 9. Summary Tables

The summary compares:

- score-balanced vs sequential at `topk=100`
- smaller sequential top-k budgets vs sequential `topk=100`
- each row vs Table 2 top-100 reference as a rough sanity check

In [ ]:
summary = results.copy()

baseline100 = (
    summary[(summary['strategy'] == 'sequential') & (summary['topk'] == 100)]
    [['dataset', 'ndcg@10', 'seconds_per_query']]
    .rename(columns={
        'ndcg@10': 'sequential_top100_ndcg@10',
        'seconds_per_query': 'sequential_top100_seconds_per_query',
    })
)

summary = summary.merge(baseline100, on='dataset', how='left')
summary['delta_vs_sequential_top100'] = summary['ndcg@10'] - summary['sequential_top100_ndcg@10']
summary['speedup_vs_sequential_top100'] = (
    summary['sequential_top100_seconds_per_query'] / summary['seconds_per_query']
)
summary['table2_note'] = (
    'Table 2 is full-query top-100; this notebook is subset-based unless MAX_QUERIES=None.'
)
summary = summary.sort_values(['dataset', 'experiment_kind', 'topk', 'strategy']).reset_index(drop=True)

summary.to_csv(SUMMARY_CSV, index=False)
SUMMARY_TXT.write_text(summary.to_string(index=False) + '\n', encoding='utf-8')

print(f'[saved] summary csv -> {SUMMARY_CSV}', flush=True)
print(f'[saved] summary txt -> {SUMMARY_TXT}', flush=True)
summary

## 10. Compact Views for Reporting

In [ ]:
grouping_view = (
    summary[summary['experiment_kind'] == 'grouping']
    .pivot_table(index='dataset', columns='strategy', values='ndcg@10', aggfunc='mean')
    .reset_index()
)
if {'sequential', 'score_balanced'}.issubset(grouping_view.columns):
    grouping_view['score_balanced_minus_sequential'] = grouping_view['score_balanced'] - grouping_view['sequential']

topk_view = (
    summary[
        summary['experiment_kind'].isin(['grouping', 'topk'])
        & summary['strategy'].eq('sequential')
    ]
    .pivot_table(index='dataset', columns='topk', values='ndcg@10', aggfunc='mean')
    .reset_index()
)

grouping_view_path = OUTPUT_DIR / 'grouping_view.csv'
topk_view_path = OUTPUT_DIR / 'topk_view.csv'
grouping_view.to_csv(grouping_view_path, index=False)
topk_view.to_csv(topk_view_path, index=False)
print(f'[saved] grouping view -> {grouping_view_path}', flush=True)
print(f'[saved] topk view -> {topk_view_path}', flush=True)

display(grouping_view)
display(topk_view)

## 11. Expansion Notes

Useful edits later:

- Add datasets in `DEFAULT_DATASETS`.
- Add score-balanced top-k variants in `TOPK_EXPERIMENTS`.
- Set `MAX_QUERIES=None` when you need a full Table 2-style run.
- Increase or reduce `PRINT_EVERY_FORWARDS` depending on how noisy Kaggle logs should be.